# Plan 6 Phase B — Colab Single-GPU Run (L4 24GB / A100 40–80GB)

Plan 6 lives in `plan_6_stock_volatility/`. This notebook clones or pulls `main`, verifies a Colab **NVIDIA L4 (24GB)** or **NVIDIA A100 (40GB / 80GB)** runtime, installs the Phase B GPU stack (`transformers>=5.10` with native `gemma4_unified` & `qwen3_5` support, `peft`, `bitsandbytes`, `accelerate`, `chronos-forecasting`), runs `scripts/colab_phase_b.py --allow-weight-download` against the **frozen Phase A bundle** (`data/frozen/`, read-only), prints the 2025 H3 comparison table, and packages `outputs/phase_b/` for download back into the repository.

- **Automatic VRAM-aware LLM selection (`--llm-model auto`)**:
  - **Colab L4 (24GB VRAM, `< 35 GB`)**: automatically selects **`google/gemma-4-12B-it`** (`11.96B` ungated `Gemma4UnifiedForConditionalGeneration`, `~11–13 GB` peak VRAM, zero OOM; override to `google/gemma-4-12B` or `Qwen/Qwen3-14B` if desired).
  - **Colab A100 (40GB / 80GB VRAM, `>= 35 GB`)**: automatically selects **`Qwen/Qwen3.8-27B`** (`Qwen3_5ForConditionalGeneration`, ungated; optional `Qwen/Qwen3-32B` or `google/gemma-4-31B-it`).
- See `PHASE_B.md` for full specifications and post-Colab CS482 final PDF integration (`scripts/07_integrate_phase_b_report.py`).

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "https://github.com/Lc-wing/data_science_project.git"
BRANCH = "main"
PLAN_DIR = "plan_6_stock_volatility"
ENTRY = "scripts/colab_phase_b.py"

def git(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)

top = Path("data_science_project").resolve()
if (top / ".git").exists():
    for args in (("fetch", "origin", BRANCH), ("checkout", BRANCH), ("pull", "origin", BRANCH)):
        done = git(*args, cwd=top)
        assert done.returncode == 0, done.stderr
else:
    done = git("clone", "--branch", BRANCH, REPO, str(top))
    assert done.returncode == 0, done.stderr

plan_root = top / PLAN_DIR
assert (plan_root / ENTRY).exists(), f"{ENTRY} not found under {plan_root}"
os.chdir(plan_root)
print("Working directory:", Path.cwd())


In [ ]:
import torch

assert torch.cuda.is_available(), "Switch Colab runtime to NVIDIA L4 (24GB) or A100 GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
default_llm = "Qwen/Qwen3.8-27B" if vram_gb >= 35.0 else "google/gemma-4-12B-it"
print(f"Detected GPU: {gpu_name} ({vram_gb:.1f} GB VRAM) -> auto LLM: {default_llm}")

# Install Phase B GPU stack (transformers>=5.10 for gemma4_unified & qwen3_5 support)
!pip install -q -U "transformers>=5.10" peft bitsandbytes accelerate chronos-forecasting
!pip install -q -U pyarrow lightgbm arch shap
!pip install -q -e . --no-deps


In [ ]:
import os

os.environ["MPLBACKEND"] = "Agg"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Override to "google/gemma-4-12B" or "Qwen/Qwen3-14B" if desired; "auto" picks L4/A100 best
LLM_MODEL = "auto"

# Run full Phase B: PatchTST 1M/15M + Chronos + FinBERT + LLM Zero-Shot & 4-bit NF4 QLoRA
!python scripts/colab_phase_b.py --allow-weight-download --llm-model {LLM_MODEL} --llm-epochs 1


In [ ]:
import shutil
from pathlib import Path

print(Path("outputs/phase_b/phase_b_comparison.md").read_text())
archive = shutil.make_archive(
    "phase_b_gpu_outputs", "zip", root_dir=".", base_dir="outputs/phase_b"
)
print("Packaged archive for download ->", archive)
